# 04 · Orchestrator–Worker with LangChain

A **planner (orchestrator)** LLM breaks a goal into subtasks, delegates each to a **worker** chain, then a **synthesizer** merges the results.

```
goal ─► [orchestrator: plan] ─► [worker ×N in parallel] ─► [synthesizer] ─► final
```

Unlike prompt chaining (fixed steps) the *number* and *content* of subtasks is decided at runtime by the model.

---

## Setup

In [1]:
%pip install -qU langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.5)
parser = StrOutputParser()

---
## 1. The orchestrator — plan into structured subtasks

We force the planner to return a **typed** plan with `with_structured_output`, so we get real Python objects instead of parsing free text.

In [3]:
class SubTask(BaseModel):
    title: str = Field(description="Short title of the subtask")
    instruction: str = Field(description="A specific, self-contained instruction for the worker")

class Plan(BaseModel):
    subtasks: List[SubTask] = Field(description="3-5 focused, non-overlapping subtasks")

planner = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a planning orchestrator. Break the goal into 3-5 focused, "
                   "non-overlapping subtasks, each independently completable by a specialist writer."),
        ("human", "Goal: {goal}"),
    ])
    | llm.with_structured_output(Plan)
)

In [4]:
GOAL = "Write a practical guide to caching strategies in high-traffic web APIs."

plan = planner.invoke({"goal": GOAL})
for i, st in enumerate(plan.subtasks, 1):
    print(f"{i}. {st.title}\n   {st.instruction}\n")

1. Introduction to Caching in Web APIs
   Write an overview explaining what caching is, its importance in high-traffic web APIs, and the basic benefits it provides.

2. Types of Caching Strategies
   Describe different caching strategies applicable to web APIs, such as in-memory caching, CDN caching, and reverse proxy caching, including their use cases and advantages.

3. Implementing Effective Caching Policies
   Provide guidance on how to design caching policies, including cache expiration, cache invalidation, and cache control headers, tailored for high-traffic scenarios.

4. Best Practices and Common Pitfalls
   Outline best practices for caching in high-traffic web APIs and warn about common pitfalls and how to avoid them.

5. Case Studies and Examples
   Include real-world examples or case studies demonstrating successful caching strategies in high-traffic web API environments.



---
## 2. The worker — one focused chain, run once per subtask

A single worker chain handles any subtask. We fan it out over all subtasks with `.batch()` so they run concurrently (see [Lesson 02](../02_parallelization/)).

In [5]:
worker = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a specialist technical writer. Complete ONLY your assigned subtask "
                   "in 1-2 tight paragraphs. Assume other sections cover the rest."),
        ("human", "Overall goal: {goal}\n\nSubtask: {title}\n{instruction}"),
    ])
    | llm | parser
)

worker_inputs = [
    {"goal": GOAL, "title": st.title, "instruction": st.instruction}
    for st in plan.subtasks
]
sections = worker.batch(worker_inputs)   # runs all workers concurrently

print(f"Completed {len(sections)} sections.")
print(sections[0][:300], "...")

Completed 5 sections.
Caching in web APIs involves storing copies of data or responses temporarily to reduce the need for repeated processing or data retrieval from primary sources. In high-traffic environments, caching is crucial for enhancing performance, reducing latency, and alleviating server load, thereby enabling  ...


---
## 3. The synthesizer — aggregate into one coherent output

In [6]:
synthesizer = (
    ChatPromptTemplate.from_messages([
        ("system", "You are an editor. Merge the drafted sections into one coherent guide: "
                   "remove redundancy, add smooth transitions, keep a logical order."),
        ("human", "Goal: {goal}\n\nDrafted sections:\n{draft}"),
    ])
    | llm | parser
)

draft = "\n\n".join(f"## {st.title}\n{sec}" for st, sec in zip(plan.subtasks, sections))
final = synthesizer.invoke({"goal": GOAL, "draft": draft})
print(final)

# Practical Guide to Caching Strategies in High-Traffic Web APIs

In high-traffic web APIs, efficient caching is essential to deliver fast, reliable, and scalable services. Proper caching reduces server load, minimizes latency, and improves user experience by serving data quickly and efficiently. This guide explores the fundamental concepts, strategies, best practices, and real-world examples to help developers implement effective caching mechanisms tailored for demanding environments.

## Introduction to Caching in Web APIs

Caching involves storing copies of data or responses temporarily to prevent redundant processing or data retrieval from primary sources. In high-traffic scenarios, caching plays a pivotal role in enhancing performance and scalability. By strategically caching responses, APIs can handle more requests with lower latency, reduce bandwidth consumption, and maintain responsiveness under heavy load. Implementing effective caching policies ensures that users receive time

---
## 4. Wrap it up as one reusable function

Plan → fan-out workers → synthesize, end to end.

In [7]:
def orchestrate(goal: str) -> str:
    plan = planner.invoke({"goal": goal})
    print(f"Orchestrator planned {len(plan.subtasks)} subtasks: "
          f"{[st.title for st in plan.subtasks]}")
    inputs = [{"goal": goal, "title": s.title, "instruction": s.instruction} for s in plan.subtasks]
    sections = worker.batch(inputs)
    draft = "\n\n".join(f"## {s.title}\n{sec}" for s, sec in zip(plan.subtasks, sections))
    return synthesizer.invoke({"goal": goal, "draft": draft})

print(orchestrate("Explain how to onboard a junior engineer to a large codebase.")[:600], "...")

Orchestrator planned 4 subtasks: ['Prepare onboarding documentation', 'Develop a guided walkthrough', 'Set up mentorship and support channels', 'Create initial tasks and small projects']
# Onboarding a Junior Engineer to a Large Codebase

Effective onboarding of a junior engineer is crucial for their success and integration into your development team. A structured approach that combines comprehensive documentation, guided walkthroughs, mentorship, and practical tasks will help them navigate the complexity of a large codebase confidently. Below is a step-by-step guide to facilitate this process.

## 1. Prepare Comprehensive Onboarding Documentation

Start by creating detailed onboarding materials that lay the foundation for understanding the codebase. This documentation should  ...


---
## What to remember

| Piece | Role |
|---|---|
| Orchestrator | LLM + `with_structured_output(Plan)` → a typed, runtime-decided task list |
| Worker | One general chain, fanned out with `.batch()` (concurrent) |
| Synthesizer | LLM that merges section drafts into a coherent whole |
| Structured output | Turns "plan" text into real objects you can loop over safely |

**vs. routing (Lesson 03):** routing picks *one* existing chain; the orchestrator *invents* the subtasks and runs *many* workers.

**Next:** 05 — **Evaluator–optimizer**: generate, score against criteria, and loop until it's good enough.